# Zadanie 3: optymalizacja dyskretna

Termin realizacji: 20 kwietnia 2026

Zadanie do oddania przez MS Teams. Do oddania: kod oraz krótkie sprawozdanie w PDF (można na przykład przy użyciu `quarto render notebook.ipynb --to pdf`).

## Na 3.0

Do realizacji:

1. Zaimplementuj dyskretny problem plecakowy z trzema plecakami w MiniZinc na podstawie przykładu (plik `minizinc.ipynb`). Spróbuj rozwiązać problem dla 10 zestawów parametrów o różnych wielkościach tak, aby rozwiązanie największego problemu trwało powyżej 5 sekund. Zanotuj w każdym przypadku liczbę wszystkich przedmiotów, pojemności plecaków, liczbę wybranych przedmiotów i sumaryczną wartość przedmiotów w każdym plecaku osobno.
2. Losowanie przedmiotów w każdym przypadku powinno być tak ustawione, aby optymalne rozwiązanie wymagało wzięcia przynajmniej dwóch przedmiotów do każdego plecaka oraz zostawienia przynajmniej dwóch przedmiotów poza plecakami. W raporcie należy umieścić informację w jaki sposób zostało to zweryfikowane.
3. Zmodyfikuj metodę z notatnika `tabu_search.ipynb` tak aby rozwiązywała opisywany problem plecakowy. Porównaj na tych samych problemach czy Minizinc i Tabu search zwracają równie dobre rozwiazania, oraz wypisz jakie to są rozwiązania. Wykonaj eksperymenty z trzema różnymi długościami listy zakazów (1, 2, 5).

## Na 4.0

Do realizacji:

1. Punkty z zadania na 3.0.
2. Rozszerz możliwe ruchy w tabu search o przeniesienie przedmiotu z jednego plecaka do drugiego. Zapisz rozważ czy to poprawia działanie metody (czy znalezione jest lepsze, takie samo czy gorsze rozwiązanie? czy rozwiązanie jest znajdowane szybciej czy wolniej?). Dla każdego z 10 zestawów parametrów problemu plecakowego wykonaj ocenę przez uśrednienie dla 10 różnych losowych przypadków.
3. Podsumuj dane w formie tabelki z czterema kolumnami (Minizinc, tabu search z listą o długości 1, 2, i 5) oraz 10 wierszami (po jednym dla zestawu parametrów problemu), a w komórkach umieść średnią wartość wartości przedmiotów oraz średni czas potrzebny do uzyskania rozwiązania.

## Na 5.0

Do realizacji:

1. Punkty z zadania na 4.0.
2. Zaimplementuj samodzielnie algorytm symulowanego wyżarzania z podobnym interfejsem co tabu search. Porównaj jego działanie do rozważanych wcześniej rozwiązań dla trzech różnych schematów chłodzenia. Dobierz liczbę iteracji tak, aby czas działania był porównywalny do tabu search.


In [4]:
import Pkg
Pkg.add("Distributions")
Pkg.add("JuMP")
Pkg.add("MiniZinc")
Pkg.add("MathOptInterface")
Pkg.add("DataStructures")
Pkg.add("Distributions")

   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
   Resolving package versions...
   Installed DataStructures ─ v0.19.4
    Updating `~/.julia/environments/v1.12/Project

In [24]:
using Distributions

function make_dzn(n::Int, capacity::Int, id::Int)
    profits = rand(DiscreteUniform(10, 1000), n)
    weights = rand(DiscreteUniform(10, 100), n)
    content = """
    ITEM = _(1..$n);
    capacity = $capacity;
    profits = $profits;
    weights = $weights;
    """
    file = open("knapsack_$id.dzn", "w+")
    write(file, content)
    close(file)
end

make_dzn (generic function with 1 method)

In [25]:
n_items = [10, 10, 10, 50, 50, 100]

for (id, i) in enumerate(n_items)
    make_dzn(i, 10 ,id)
end

In [51]:
for i in 1:length(n_items)
    result = read(`minizinc --solver gecode my_knapsack.mzn knapsack_$i.dzn` , String)
    println(result)
end

taken = [true, true, true, true, true, true, true, true, true, true];
profit = 6058;
----------

taken = [true, true, true, true, true, true, true, true, true, true];
profit = 5694;
----------

taken = [true, true, true, true, true, true, true, true, true, true];
profit = 5319;
----------

taken = [true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true];
profit = 22970;
----------

taken = [true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true];
profit = 23569;
----------

taken = [true, true, true, true, 

## Symulowane Wyżarzanie

In [ ]:
using DataStructures
using Distributions

mutable struct SAState{P, TF}
    best_seen::P
    best_seen_obj::TF
    current::P
    current_obj::TF
    considered::P
    temp::Float64
    iter::Int
end

function SAState(p, x0; initial_temp::Float64=100.0)
    obj = objective(p, x0)
    return SAState{typeof(x0), typeof(obj)}(
        copy(x0), obj, copy(x0), obj, copy(x0), initial_temp, 1
    )
end

function solve_sa(p, s::SAState; iteration_limit::Int=1000, cooling_rate::Float64=0.95)
    while s.iter < iteration_limit
        move = random_move(p, s.current) 
        
        copyto!(s.considered, s.current)
        apply!(s.considered, move)
        trial_obj = objective(p, s.considered)
    
        ΔE = trial_obj - s.current_obj
        
        if ΔE < 0 || rand() < exp(-ΔE / s.temp)
            copyto!(s.current, s.considered)
            s.current_obj = trial_obj
            
            if s.current_obj < s.best_seen_obj
                copyto!(s.best_seen, s.current)
                s.best_seen_obj = s.current_obj
            end
        end
        
        s.temp *= cooling_rate
        s.iter += 1
    end
    return s.best_seen
end


struct KnapsackProblem
    capacity::Int
    weights::Vector{Int}
    profits::Vector{Int}
end

function random_move(p::KnapsackProblem, x::Vector{Bool})
    i = rand(1:length(x))
    
    current_weight = sum(p.weights .* x)
    
    if x[i]
        return (:remove, i)
    else
        if current_weight + p.weights[i] <= p.capacity
            return (:add, i)
        else
            in_bag_indices = findall(x)
            if isempty(in_bag_indices)
                return (:add, i)
            end
            return (:remove, rand(in_bag_indices))
        end
    end
end

function objective(p::KnapsackProblem, x)
    return -sum(p.profits .* x)
end


function apply!(x, move::Tuple{Symbol,Int})
    if move[1] === :add
        x[move[2]] = true
    else
        x[move[2]] = false
    end
    return x
end

function invert_move(::KnapsackProblem, move::Tuple{Symbol,Int})
    if move[1] === :add
        return (:remove, move[2])
    else
        return (:add, move[2])
    end
end


function possible_moves(p::KnapsackProblem, x::Vector{Bool})
    move_list = Tuple{Symbol,Int}[]
    current_weight = sum(p.weights .* x)
    # add item
    for i in eachindex(x, p.weights)
        if !x[i] && current_weight + p.weights[i] <= p.capacity
            push!(move_list, (:add, i))
        end
    end
    # remove item
    for i in eachindex(x, p.weights)
        if x[i]
            push!(move_list, (:remove, i))
        end
    end
    return move_list
end



possible_moves (generic function with 1 method)

In [6]:
function generate_problem()
    n_items = 100
    profits = rand(DiscreteUniform(10, 1000), n_items)
    weights = rand(DiscreteUniform(10, 100), n_items)
    kp = KnapsackProblem(3000, profits, weights)
end

kp1 = generate_problem()


KnapsackProblem(3000, [533, 326, 924, 913, 760, 574, 532, 283, 163, 756  …  781, 313, 634, 25, 621, 598, 867, 539, 519, 867], [65, 10, 66, 74, 86, 31, 69, 13, 55, 79  …  42, 33, 93, 38, 98, 82, 62, 17, 83, 13])

In [9]:

function test(kp)
    x0 = fill(false, length(kp.weights))
    st = SAState(kp, x0; buffer_length=10)
    sol = solve_sas(kp, st; iteration_limit=1000000)
    println(findall(sol))
    println("Best objective: ", st.best_seen_obj)
    println("Last iteration: ", st.iter)
end

test(kp1)

MethodError: MethodError: no method matching SAState(::KnapsackProblem, ::Vector{Bool}; buffer_length::Int64)
This method does not support all of the given keyword arguments (and may not support any).

Closest candidates are:
  SAState(::Any, ::Any; initial_temp) got unsupported keyword argument "buffer_length"
   @ Main ~/Desktop/Metody-i-algorytmy-optymalizacji/lab03/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W6sZmlsZQ==.jl:14
  SAState(::P, ::TF, !Matched::P, !Matched::TF, !Matched::P, !Matched::Float64, !Matched::Int64) where {P, TF} got unsupported keyword argument "buffer_length"
   @ Main ~/Desktop/Metody-i-algorytmy-optymalizacji/lab03/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W6sZmlsZQ==.jl:5
